In [2]:
import os
import pandas as pd
import json
import shutil
from google.colab import files

In [3]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
categorias = pd.read_csv("categorias.csv", sep=";", encoding="latin1")
productos = pd.read_csv("productos.csv", sep=";", encoding="latin1")
clientes = pd.read_csv("clientes.csv", sep=";", encoding="latin1")
movimientos_stock = pd.read_csv("movimientos_stock.csv", sep=";", encoding="latin1")
ventas = pd.read_csv("ventas.csv", sep=";", encoding="latin1")
marcas = pd.read_csv("marcas.csv", sep=";", encoding="latin1")


In [ ]:
marcas.head()

In [ ]:
ventas.head()

In [ ]:
productos.head()

In [ ]:
clientes.head()

In [ ]:
movimientos_stock.head()

In [ ]:
categorias.head()

In [ ]:
print("Categorias:",categorias.shape)
print("Productos:",productos.shape)
print("Clientes:",clientes.shape)
print("Movimientos Stock:",movimientos_stock.shape)
print("Ventas:",ventas.shape)
print("Marcas:",marcas.shape)

In [ ]:
print("Columnas Categorias:")
print(categorias.columns)
print("Columnas Productos:")
print(productos.columns)
print("Columnas Clientes:")
print(clientes.columns)
print("Columnas Movimientos Stock:")
print(movimientos_stock.columns)
print("Columnas Ventas:")
print(ventas.columns)
print("Columnas Marcas:")
print(marcas.columns)

In [ ]:
categorias.columns = categorias.columns.str.strip()
productos.columns = productos.columns.str.strip()
clientes.columns = clientes.columns.str.strip()
movimientos_stock.columns = movimientos_stock.columns.str.strip()
ventas.columns = ventas.columns.str.strip()
marcas.columns = marcas.columns.str.strip()


In [ ]:
categorias.isnull().sum()

In [ ]:
productos.isnull().sum()

In [ ]:
clientes.isnull().sum()

In [ ]:
movimientos_stock.isnull().sum()

In [ ]:
ventas.isnull().sum()

In [ ]:

marcas.isnull().sum()

In [ ]:
categorias["tipo_categoria"] = categorias["id_padre"].apply(
    lambda x: "PADRE" if pd.isnull(x) or x == 0 else "HIJA"
)

In [ ]:
print("Duplicados categorias:",categorias.duplicated().sum())
print("Duplicados productos:",productos.duplicated().sum())
print("Duplicados clientes:",clientes.duplicated().sum())
print("Duplicados Movimientos Stock:",movimientos_stock.duplicated().sum())
print("Duplicados Ventas:",ventas.duplicated().sum())
print("Duplicados Marcas",marcas.duplicated().sum())

In [ ]:
categorias = categorias.drop_duplicates()
productos = productos.drop_duplicates()
clientes = clientes.drop_duplicates()
movimientos_stock = movimientos_stock.drop_duplicates()
ventas = ventas.drop_duplicates()
marcas = marcas.drop_duplicates()

In [ ]:
ventas['fecha'] = pd.to_datetime(ventas['fecha'],errors="coerce")

In [ ]:
ventas["anio"] = ventas['fecha'].dt.year
ventas["mes"] = ventas["fecha"].dt.month
ventas["dia"] = ventas['fecha'].dt.day
ventas["periodo"] = ventas['fecha'].dt.to_period("M").astype(str)

In [ ]:
productos['descripcion'] = productos['descripcion'].astype(str).str.strip().str.upper()

In [ ]:
ventas.head()

In [ ]:
import pandas as pd
import json

# 1. Convertimos la columna productos de texto JSON a lista real
ventas["productos_lista"] = ventas["productos"].apply(json.loads)

# 2. Creamos una tabla donde cada producto vendido sea una fila
detalle_ventas = ventas[[
    "id",
    "codigo",
    "id_cliente",
    "fecha",
    "metodo_pago",
    "productos_lista"
]].explode("productos_lista")

# 3. Convertimos el diccionario de cada producto en columnas
productos_detalle = pd.json_normalize(detalle_ventas["productos_lista"])

# 4. Renombramos columnas para evitar confusión
detalle_ventas = detalle_ventas.rename(columns={
    "id": "id_venta"
})

productos_detalle = productos_detalle.rename(columns={
    "id": "id_producto",
    "total": "subtotal_producto"
})

# 5. Reiniciamos índices en ambas tablas
detalle_ventas = detalle_ventas.drop(columns=["productos_lista"]).reset_index(drop=True)
productos_detalle = productos_detalle.reset_index(drop=True)

# 6. Concatenamos de lado a lado
detalle_ventas = pd.concat(
    [detalle_ventas, productos_detalle],
    axis=1
)

# 7. Mostramos resultado
detalle_ventas.head()

In [ ]:
detalle_ventas.shape

In [ ]:
detalle_ventas.columns

In [ ]:
detalle_ventas['fecha'] = pd.to_datetime(detalle_ventas['fecha'],errors="coerce")
columnas_numericas = [
  'id_venta', 'codigo', 'id_cliente',
       'id_producto', 'cantidad', 'stock', 'precio',
       'subtotal_producto'
]
for col in columnas_numericas:
  detalle_ventas[col] = pd.to_numeric(detalle_ventas[col],errors="coerce")

In [ ]:
detalle_ventas.dtypes

In [ ]:
detalle_ventas.isnull().sum()

In [ ]:
detalle_ventas.duplicated().sum()

In [ ]:
detalle_ventas.describe()

In [ ]:
detalle_ventas["cantidad_docenas"] = detalle_ventas["cantidad"]/12

In [ ]:
detalle_ventas["anio"]= detalle_ventas["fecha"].dt.year
detalle_ventas["mes"]= detalle_ventas["fecha"].dt.month
detalle_ventas["dia"]= detalle_ventas["fecha"].dt.day
detalle_ventas["periodo"]= detalle_ventas["fecha"].dt.to_period("M").astype(str)

In [ ]:
detalle_ventas["subtotal_calculado"] = (detalle_ventas["cantidad_docenas"]*detalle_ventas["precio"].round(2))

detalle_ventas["diferencia_subtotal"] = (detalle_ventas["subtotal_calculado"]-detalle_ventas['subtotal_producto']).round(2)

In [ ]:
detalle_ventas.head()

In [ ]:
for nombre in ["categorias", "productos", "clientes", "movimientos_stock", "ventas", "marcas", "detalle_ventas"]:
    if nombre in globals():
      print(nombre,globals()[nombre].shape)
    else:
      print(nombre,"no existe")

In [ ]:
dim_categorias = categorias.copy().rename(columns={
    "id": "id_categoria",
    "categoria": "categoria"
})
dim_productos = productos.copy().rename(columns={
    "id": "id_producto",
    "descripcion": "producto"
})
dim_marcas = marcas.copy().rename(columns={
    "id": "id_marca",
    "marca": "marca"
})
dim_clientes = clientes.copy().rename(columns={
    "id": "id_cliente",
    "nombre": "cliente"
})
fact_ventas = ventas.copy().rename(columns={
    "id": "id_venta",
    "total": "total_venta",
    "neto": "neto_venta"
})

fact_detalle_ventas = detalle_ventas.copy()

fact_movimientos_stock = movimientos_stock.copy().rename(columns={
    "id": "id_movimiento",

})


In [ ]:
fact_ventas = fact_ventas.drop(
    columns=["productos", "producto_lista", "productos_lista"],
    errors="ignore"
)

In [ ]:
fact_detalle_ventas.head()

In [ ]:

fact_movimientos_stock["fecha"] = pd.to_datetime(fact_movimientos_stock["fecha"], errors="coerce")
fact_movimientos_stock["periodo"] = fact_movimientos_stock["fecha"].dt.to_period("M").astype(str)





In [ ]:
dim_clientes_bi = dim_clientes[[
    "id_cliente",
    "cliente",
    "ciudad",
    "compras",
    "ultima_compra",
    "fecha"
]].copy()



In [ ]:
fact_ventas["estado_venta"]= fact_ventas['estado'].apply(lambda x:"ACTIVA" if x==1 else "ANULADA")

In [ ]:
fact_ventas[fact_ventas["tipo_cambio"] > 10]

In [ ]:
fact_ventas["tipo_cambio_observacion"] = fact_ventas["tipo_cambio"].apply(
    lambda x: "REVISAR" if x > 10 else "OK"
)

In [ ]:
fact_detalle_ventas["producto_existe_en_maestro"] = fact_detalle_ventas["id_producto"].isin(
    dim_productos["id_producto"]
)

In [ ]:
fact_detalle_ventas[fact_detalle_ventas["precio"] > 1000]

In [ ]:
fact_detalle_ventas["precio_observacion"] = fact_detalle_ventas["precio"].apply(
    lambda x: "REVISAR" if x > 1000 else "OK"
)

In [ ]:
fact_detalle_ventas[
    fact_detalle_ventas["diferencia_subtotal"] != 0
]

In [ ]:
fact_detalle_ventas["subtotal_observacion"] = fact_detalle_ventas["diferencia_subtotal"].apply(
    lambda x: "REVISAR" if x != 0 else "OK"
)

In [ ]:
fact_movimientos_stock = fact_movimientos_stock.drop(
    columns=["precio_unitario"],
    errors="ignore"
)

In [ ]:
dim_categorias = dim_categorias[dim_categorias["categoria"] != "ZAPATILLAS REGIMEN GENERAL"]
dim_categorias = dim_categorias[dim_categorias["categoria"] != "CASACAS REGIMEN GENERAL"]


In [ ]:
dim_categorias["id_padre"] = dim_categorias["id_padre"].astype("Int64")
mapa_categorias = {
    15: "CASACAS",
    16: "ZAPATILLAS",

}
dim_categorias["categoria_principal"] = dim_categorias["id_padre"].map(mapa_categorias)
dim_categorias.head()

In [ ]:
dim_categorias.head()

In [ ]:
fact_detalle_ventas.head()

In [ ]:
dim_productos.head()

In [ ]:
dim_productos = dim_productos.drop(columns=["imagen"])

In [ ]:
movimientos_stock = movimientos_stock.drop(columns=["precio_unitario"])

In [ ]:
movimientos_stock.isnull().all()


In [ ]:
dim_clientes.head()

In [ ]:
dim_clientes = dim_clientes.drop(columns=["direccion", "documento", "id_tipo_doc"])

In [ ]:
dim_clientes = dim_clientes.drop(columns=["razon_social"])

In [ ]:
dim_clientes["pais"] =  dim_clientes["pais"].fillna("Peru")

In [ ]:

carpeta_salida = "tablas_powerbi_final"
os.makedirs(carpeta_salida, exist_ok=True)

# =========================
# 2. Lista de tablas finales a exportar
# =========================

tablas_finales = {
    "dim_categorias": dim_categorias,
    "dim_marcas": dim_marcas,
    "dim_productos": dim_productos,
    "dim_clientes_bi": dim_clientes_bi,
    "fact_ventas": fact_ventas,
    "fact_detalle_ventas": fact_detalle_ventas,
    "fact_movimientos_stock": fact_movimientos_stock
}

# =========================
# 3. Exportar cada tabla como CSV
# =========================

for nombre, tabla in tablas_finales.items():
    ruta = f"{carpeta_salida}/{nombre}.csv"
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Exportado: {nombre} -> {tabla.shape}")

# =========================
# 4. Crear archivo ZIP
# =========================

shutil.make_archive("tablas_powerbi_final", "zip", carpeta_salida)

print("Archivo ZIP creado correctamente.")

# =========================
# 5. Descargar ZIP
# =========================

files.download("tablas_powerbi_final.zip")

In [ ]:

tablas_finales = {
    "dim_categorias": dim_categorias,
    "dim_marcas": dim_marcas,
    "dim_productos": dim_productos,
    "dim_clientes_bi": dim_clientes_bi,
    "fact_ventas": fact_ventas,
    "fact_detalle_ventas": fact_detalle_ventas,
    "fact_movimientos_stock": fact_movimientos_stock
}

nombre_archivo = "tablas_powerbi_final.xlsx"

with pd.ExcelWriter(nombre_archivo, engine="openpyxl") as writer:
    for nombre, tabla in tablas_finales.items():
        tabla.to_excel(writer, sheet_name=nombre[:31], index=False)

files.download(nombre_archivo)

df = pd.read_csv("data_productos_bi.csv",encoding="utf-8-sig")

In [7]:
import pandas as pd

df = pd.read_csv("data_productos_bi.csv",encoding = "utf-8-sig")
df.head()

,Suma de bultos,Suma de cantidad_bulto,codigo,Año,Trimestre,Mes,Día,Año.1,Trimestre.1,Mes.1,...,id_categoria,id_marca,id_producto,observacion,Suma de precio_compra,producto,Suma de precio_venta,Suma de stock,Suma de stock_inicial,Suma de ventas
0,58.0,2.0,CASACA DAMAA,2025,Qtr 3,julio,2,NaN,NaN,NaN,...,1,1,1977,CASA,200,1.1,200,101,114.0,13.0
1,100.0,1.0,CASACA VARON,2025,Qtr 3,julio,2,NaN,NaN,NaN,...,2,1,1976,BOLETAS UNIDADES,50,1-1,60,47,48.0,1.0
2,NaN,NaN,8901,2022,Qtr 3,septiembre,5,NaN,NaN,NaN,...,11,4,370,NaN,157,175-190,195,182,NaN,264.0
3,NaN,NaN,V6573712M,2022,Qtr 2,mayo,30,NaN,NaN,NaN,...,2,1,109,NaN,210,210,230,0,NaN,60.0
4,1.0,60.0,11850-9,2023,Qtr 1,febrero,2,2024.0,Qtr 2,mayo,...,1,13,667,.,240,2XL A 5XL,260,0,60.0,336.0


In [ ]:
df.columns

In [8]:
productos = df[[
    "id_producto",
    "codigo",
    "producto",
    "id_categoria",
    "id_marca",
    "observacion",
    "Suma de precio_compra",
    "Suma de precio_venta",
    "Suma de stock",
    "Suma de stock_inicial",
    "Suma de ventas"
]].copy()
productos.head()

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,Suma de ventas
0,1977,CASACA DAMAA,1.1,1,1,CASA,200,200,101,114.0,13.0
1,1976,CASACA VARON,1-1,2,1,BOLETAS UNIDADES,50,60,47,48.0,1.0
2,370,8901,175-190,11,4,NaN,157,195,182,NaN,264.0
3,109,V6573712M,210,2,1,NaN,210,230,0,NaN,60.0
4,667,11850-9,2XL A 5XL,1,13,.,240,260,0,60.0,336.0


In [9]:
import unicodedata
import re

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.strip()

    return texto

productos["producto_limpio"] = productos["producto"].apply(limpiar_texto)
productos["codigo_limpio"] = productos["codigo"].apply(limpiar_texto)

productos[["codigo", "producto", "producto_limpio"]].head(10)

,codigo,producto,producto_limpio
0,CASACA DAMAA,1.1,1.1
1,CASACA VARON,1-1,1-1
2,8901,175-190,175-190
3,V6573712M,210,210
4,11850-9,2XL A 5XL,2XL A 5XL
5,B819916XL,2XL A 5XL,2XL A 5XL
6,B819917XL,2XL A 5XL,2XL A 5XL
7,B819920XL,2XL A 5XL,2XL A 5XL
8,20-05#-414,2XL-5XL,2XL-5XL
9,2508,3/4 CON FRIZA L A 3XL,3/4 CON FRIZA L A 3XL


In [10]:
from collections import Counter

palabras = []

for texto in productos["producto_limpio"]:
    palabras.extend(texto.split())

conteo_palabras = Counter(palabras)

conteo_palabras.most_common(50)

[('A', 2118),
 ('CASACA', 1334),
 ('3XL', 899),
 ('CON', 861),
 ('VARON', 826),
 ('DAMA', 726),
 ('S', 679),
 ('YD', 669),
 ('2XL', 665),
 ('CHINO', 565),
 ('M', 543),
 ('FIBRA', 423),
 ('FRIZA', 418),
 ('L', 349),
 ('REVERSIBLE', 315),
 ('4XL', 270),
 ('DE', 253),
 ('XL', 215),
 ('ZAPATILLA', 194),
 ('3/4', 189),
 ('CORTAVIENTO', 166),
 ('5XL', 151),
 ('LARGA', 149),
 ('PIEL', 145),
 ('IMPERMEABLE', 140),
 ('CHALECO', 125),
 ('CORTO', 122),
 ('PANO', 114),
 ('CUERO', 106),
 ('TELA', 105),
 ('6XL', 99),
 ('NEGRO', 96),
 ('SOLO', 84),
 ('TIPO', 82),
 ('36', 78),
 ('SIN', 74),
 ('NINO', 74),
 ('PELO', 67),
 ('COLORES', 66),
 ('PARKA', 65),
 ('39', 64),
 ('LARGO', 61),
 ('GRUESO', 60),
 ('COLOR', 60),
 ('31', 60),
 ('BLANCO', 59),
 ('CONNMODA', 56),
 ('-', 55),
 ('HOMBRE', 55),
 ('DRILL', 52)]

In [11]:
def extraer_genero(texto):
    texto = limpiar_texto(texto)

    if "DAMA" in texto or "MUJER" in texto:
        return "Dama"
    elif "VARON" in texto or "HOMBRE" in texto:
        return "Varón"
    elif "NINO" in texto:
        return "Niño"
    elif "NINA" in texto:
        return "Niña"
    elif "UNISEX" in texto:
        return "Unisex"
    else:
        return "No identificado"

productos["genero"] = productos["producto_limpio"].apply(extraer_genero)

In [12]:
def extraer_material(texto):
    texto = limpiar_texto(texto)

    if "CUERO" in texto or "CUERINA" in texto:
        return "Cuero / Cuerina"
    elif "JEAN" in texto:
        return "Jean"
    elif "POLAR" in texto:
        return "Polar"
    elif "TASLAN" in texto:
        return "Taslan"
    elif "ALGODON" in texto:
        return "Algodón"
    elif "TELA" in texto:
        return "Tela"
    elif "FIBRA" in texto:
        return "Fibra"
    elif "PANO" in texto:
        return "Paño"
    elif "DRILL" in texto:
        return "Drill"
    elif "IMPERMEABLE" in texto:
        return "Impermeable"
    else:
        return "No identificado"

productos["material"] = productos["producto_limpio"].apply(extraer_material)

In [13]:
def extraer_caracteristicas(texto):
    texto = limpiar_texto(texto)
    caracteristicas = []

    if "FRIZA" in texto or "FRISA" in texto:
        caracteristicas.append("Con friza")
    if "REVERSIBLE" in texto or "REVERSIBE" in texto:
        caracteristicas.append("Reversible")
    if "CAPUCHA" in texto:
        caracteristicas.append("Con capucha")
    if "PELO" in texto or "PELUCHE" in texto:
        caracteristicas.append("Con pelo/peluche")
    if "CALEFACCION" in texto:
        caracteristicas.append("Con calefacción")
    if "IMPERMEABLE" in texto:
        caracteristicas.append("Impermeable")
    if "LARGA" in texto or "LARGO" in texto:
        caracteristicas.append("Larga")
    if "CORTA" in texto or "CORTO" in texto:
        caracteristicas.append("Corta")
    if "GRUESA" in texto or "GRUESO" in texto:
        caracteristicas.append("Gruesa")
    if "DELGADA" in texto or "DELGADO" in texto:
        caracteristicas.append("Delgada")

    if len(caracteristicas) == 0:
        return "Sin característica identificada"

    return ", ".join(caracteristicas)

productos["caracteristicas"] = productos["producto_limpio"].apply(extraer_caracteristicas)

In [14]:
def extraer_talla(texto):
    texto = limpiar_texto(texto)

    patron = r"(XS|S|M|L|XL|XXL|2XL|3XL|4XL|5XL|6XL|7XL|8XL)\s*(A|-)\s*(XS|S|M|L|XL|XXL|2XL|3XL|4XL|5XL|6XL|7XL|8XL)"
    resultado = re.search(patron, texto)

    if resultado:
        return resultado.group(0).replace(" A ", " a ").replace("-", " a ")

    patron_simple = r"\b(XS|S|M|L|XL|XXL|2XL|3XL|4XL|5XL|6XL|7XL|8XL)\b"
    resultado_simple = re.search(patron_simple, texto)

    if resultado_simple:
        return resultado_simple.group(0)

    return "No identificado"

productos["rango_talla"] = productos["producto_limpio"].apply(extraer_talla)

In [15]:
def extraer_temporada(texto):
    texto = limpiar_texto(texto)

    if any(palabra in texto for palabra in [
        "FRIZA", "FRISA", "PELO", "PELUCHE", "POLAR", "GRUESA",
        "GRUESO", "CALEFACCION", "PANO"
    ]):
        return "Invierno"

    elif any(palabra in texto for palabra in [
        "DELGADA", "DELGADO", "TELA FINA", "CORTAVIENTO"
    ]):
        return "Media estación"

    elif any(palabra in texto for palabra in [
        "IMPERMEABLE", "TASLAN"
    ]):
        return "Lluvia / Media estación"

    else:
        return "Todo el año / Revisar"

productos["temporada_sugerida"] = productos["producto_limpio"].apply(extraer_temporada)

In [16]:
productos[[
    "codigo",
    "producto",
    "genero",
    "material",
    "caracteristicas",
    "rango_talla",
    "temporada_sugerida"
]].head(30)

,codigo,producto,genero,material,caracteristicas,rango_talla,temporada_sugerida
0,CASACA DAMAA,1.1,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
1,CASACA VARON,1-1,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
2,8901,175-190,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
3,V6573712M,210,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
4,11850-9,2XL A 5XL,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar
5,B819916XL,2XL A 5XL,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar
6,B819917XL,2XL A 5XL,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar
7,B819920XL,2XL A 5XL,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar
8,20-05#-414,2XL-5XL,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar
9,2508,3/4 CON FRIZA L A 3XL,No identificado,No identificado,Con friza,L a 3XL,Invierno


In [17]:
productos["genero"].value_counts()

,count
genero,
Varón,873
No identificado,821
Dama,720
Niño,74
Niña,23


In [18]:
productos["material"].value_counts()

,count
material,
No identificado,1489
Fibra,410
Cuero / Cuerina,155
Impermeable,142
Paño,114
Tela,105
Drill,52
Taslan,19
Algodón,13


In [19]:
productos[
    productos["material"] == "No identificado"
][["codigo", "producto"]].head(50)

,codigo,producto
0,CASACA DAMAA,1.1
1,CASACA VARON,1-1
2,8901,175-190
3,V6573712M,210
4,11850-9,2XL A 5XL
5,B819916XL,2XL A 5XL
6,B819917XL,2XL A 5XL
7,B819920XL,2XL A 5XL
8,20-05#-414,2XL-5XL
9,2508,3/4 CON FRIZA L A 3XL


In [20]:
productos[
    productos["genero"] == "No identificado"
][["codigo", "producto"]].head(50)

,codigo,producto
0,CASACA DAMAA,1.1
1,CASACA VARON,1-1
2,8901,175-190
3,V6573712M,210
4,11850-9,2XL A 5XL
5,B819916XL,2XL A 5XL
6,B819917XL,2XL A 5XL
7,B819920XL,2XL A 5XL
8,20-05#-414,2XL-5XL
9,2508,3/4 CON FRIZA L A 3XL


In [21]:
productos.to_csv("productos_enriquecidos.csv", index=False, encoding="utf-8-sig")

In [40]:
import pandas as pd
import unicodedata
import re
productos = pd.read_csv("productos_enriquecidos.csv",encoding="utf-8-sig")
categorias = pd.read_csv("data_categorias_bi.csv",encoding="utf-8-sig")

productos.head()

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,Suma de ventas,producto_limpio,codigo_limpio,genero,material,caracteristicas,rango_talla,temporada_sugerida
0,1977,CASACA DAMAA,1.1,1,1,CASA,200,200,101,114.0,13.0,1.1,CASACA DAMAA,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
1,1976,CASACA VARON,1-1,2,1,BOLETAS UNIDADES,50,60,47,48.0,1.0,1-1,CASACA VARON,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
2,370,8901,175-190,11,4,NaN,157,195,182,NaN,264.0,175-190,8901,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
3,109,V6573712M,210,2,1,NaN,210,230,0,NaN,60.0,210,V6573712M,No identificado,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar
4,667,11850-9,2XL A 5XL,1,13,.,240,260,0,60.0,336.0,2XL A 5XL,11850-9,No identificado,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar


In [41]:
productos_cat = productos.merge(categorias[["id_categoria","categoria","categoria_principal","tipo_categoria"]],
                                on="id_categoria",
                                how="left"
                                )
productos_cat[["codigo","producto","id_categoria","categoria","categoria_principal","tipo_categoria"]].head()

,codigo,producto,id_categoria,categoria,categoria_principal,tipo_categoria
0,CASACA DAMAA,1.1,1,CASACA DE DAMA,CASACAS,HIJA
1,CASACA VARON,1-1,2,CASACA DE HOMBRE,CASACAS,HIJA
2,8901,175-190,11,BLAZER VARON,CASACAS,HIJA
3,V6573712M,210,2,CASACA DE HOMBRE,CASACAS,HIJA
4,11850-9,2XL A 5XL,1,CASACA DE DAMA,CASACAS,HIJA


In [43]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.strip()

    return texto

productos_cat["producto_limpio"] = productos_cat["producto"].apply(limpiar_texto)
productos_cat["categoria_limpia"] = productos_cat["categoria"].apply(limpiar_texto)

In [44]:
def genero_desde_categoria(texto):
    texto = limpiar_texto(texto)

    if "DAMA" in texto or "MUJER" in texto:
        return "Dama"
    elif "VARON" in texto or "HOMBRE" in texto:
        return "Varón"
    elif "NINA" in texto:
        return "Niña"
    elif "NINO" in texto:
        return "Niño"
    elif "ADULTO" in texto:
        return "Varón"
    else:
        return "No identificado"

productos_cat["genero_categoria"] = productos_cat["categoria_limpia"].apply(genero_desde_categoria)

In [48]:
productos_cat.head()

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,...,caracteristicas,rango_talla,temporada_sugerida,categoria,categoria_principal,tipo_categoria,categoria_limpia,genero_categoria,genero_producto,genero_final
0,1977,CASACA DAMAA,1.1,1,1,CASA,200,200,101,114.0,...,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE DAMA,CASACAS,HIJA,CASACA DE DAMA,Dama,No identificado,Dama
1,1976,CASACA VARON,1-1,2,1,BOLETAS UNIDADES,50,60,47,48.0,...,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón,No identificado,Varón
2,370,8901,175-190,11,4,NaN,157,195,182,NaN,...,Sin característica identificada,No identificado,Todo el año / Revisar,BLAZER VARON,CASACAS,HIJA,BLAZER VARON,Varón,No identificado,Varón
3,109,V6573712M,210,2,1,NaN,210,230,0,NaN,...,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón,No identificado,Varón
4,667,11850-9,2XL A 5XL,1,13,.,240,260,0,60.0,...,Sin característica identificada,2XL a 5XL,Todo el año / Revisar,CASACA DE DAMA,CASACAS,HIJA,CASACA DE DAMA,Dama,No identificado,Dama


In [46]:
def genero_desde_producto(texto):
    texto = limpiar_texto(texto)

    if "DAMA" in texto or "MUJER" in texto:
        return "Dama"
    elif "VARON" in texto or "HOMBRE" in texto:
        return "Varón"
    elif "NINA" in texto:
        return "Niña"
    elif "NINO" in texto:
        return "Niño"
    elif "UNISEX" in texto:
        return "Unisex"
    else:
        return "No identificado"

productos_cat["genero_producto"] = productos_cat["producto_limpio"].apply(genero_desde_producto)

In [47]:
def genero_final(fila):
    if fila["genero_categoria"] != "No identificado":
        return fila["genero_categoria"]
    elif fila["genero_producto"] != "No identificado":
        return fila["genero_producto"]
    else:
        return "No identificado"

productos_cat["genero_final"] = productos_cat.apply(genero_final, axis=1)

In [49]:
productos_cat[[
    "codigo",
    "producto",
    "categoria",
    "categoria_principal",
    "genero_categoria",
    "genero_producto",
    "genero_final"
]].head(30)

,codigo,producto,categoria,categoria_principal,genero_categoria,genero_producto,genero_final
0,CASACA DAMAA,1.1,CASACA DE DAMA,CASACAS,Dama,No identificado,Dama
1,CASACA VARON,1-1,CASACA DE HOMBRE,CASACAS,Varón,No identificado,Varón
2,8901,175-190,BLAZER VARON,CASACAS,Varón,No identificado,Varón
3,V6573712M,210,CASACA DE HOMBRE,CASACAS,Varón,No identificado,Varón
4,11850-9,2XL A 5XL,CASACA DE DAMA,CASACAS,Dama,No identificado,Dama
5,B819916XL,2XL A 5XL,CHALECO DAMA,CASACAS,Dama,No identificado,Dama
6,B819917XL,2XL A 5XL,CHALECO DAMA,CASACAS,Dama,No identificado,Dama
7,B819920XL,2XL A 5XL,CHALECO DAMA,CASACAS,Dama,No identificado,Dama
8,20-05#-414,2XL-5XL,CASACA DE HOMBRE,CASACAS,Varón,No identificado,Varón
9,2508,3/4 CON FRIZA L A 3XL,CASACA DE DAMA,CASACAS,Dama,No identificado,Dama


In [35]:
productos_cat[
    productos_cat["genero_final"] == "No identificado"
][["codigo", "producto", "categoria"]].head(50)

,codigo,producto,categoria
717,CH J109,CASACA JEAN LARGO,CASACA CHINO
1567,ZGD2309BG4,CHIMUN CONNMODA 36 A 41,ZAPATILLAS CHIMPUN
1709,CODIGO PRUEBA,DESCRIPCION PRUEBA,CATEGORIA PRUEBA
2333,B8265,ZAPATILLA DE BASQUE 36 A 40 CHAORI CON 2 COLOR,ZAPATILLA JUVENIL BASQUET
2334,22605-RG,ZAPATILLA DE BASQUET 36 A 40,ZAPATILLA JUVENIL BASQUET
2335,8217-RG,ZAPATILLA DE BASQUET 36 A 40 CON 2 COLORES Y R...,ZAPATILLA JUVENIL BASQUET
2336,7576-RG,ZAPATILLA DE BASQUET 36 A 40 CON PEGA CON 2 COLOR,ZAPATILLA JUVENIL BASQUET
2337,YSANLU2505-014,ZAPATILLA DE BASQUET JUVENIL 36 A 40 CON REGUL...,ZAPATILLA JUVENIL BASQUET
2338,B8271,ZAPATILLA DE BASQUET SOLO CUERO 36 A 40 CHAORI,ZAPATILLA JUVENIL BASQUET
2360,MTR150-JU,ZAPATILLA DE FUTBOL JUVENIL CHOTERA BOLKA 36 A 41,ZAPATILLAS DE FUTBOL JUVENIL


In [57]:
productos_limpieza_1 = productos_cat.drop(columns=["genero_producto","genero_categoria"])




In [65]:
productos_f=productos_limpieza_1.drop(columns=["genero"])

In [66]:
productos_f

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,...,codigo_limpio,material,caracteristicas,rango_talla,temporada_sugerida,categoria,categoria_principal,tipo_categoria,categoria_limpia,genero_final
0,1977,CASACA DAMAA,1.1,1,1,CASA,200,200,101,114.0,...,CASACA DAMAA,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE DAMA,CASACAS,HIJA,CASACA DE DAMA,Dama
1,1976,CASACA VARON,1-1,2,1,BOLETAS UNIDADES,50,60,47,48.0,...,CASACA VARON,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
2,370,8901,175-190,11,4,NaN,157,195,182,NaN,...,8901,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,BLAZER VARON,CASACAS,HIJA,BLAZER VARON,Varón
3,109,V6573712M,210,2,1,NaN,210,230,0,NaN,...,V6573712M,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
4,667,11850-9,2XL A 5XL,1,13,.,240,260,0,60.0,...,11850-9,No identificado,Sin característica identificada,2XL a 5XL,Todo el año / Revisar,CASACA DE DAMA,CASACAS,HIJA,CASACA DE DAMA,Dama
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2506,2341,YFD6321G6W,ZAPATILLA SOLO BLANCO CONNMODA 36 A 41 TEJIDO,18,21,4TO PISO,65,75,12,12.0,...,YFD6321G6W,No identificado,Sin característica identificada,LA S,Todo el año / Revisar,ZAPATILLA DE ADULTO,ZAPATILLAS,HIJA,ZAPATILLA DE ADULTO,Varón
2507,2340,N22115C-N,ZAPATILLA TACO MODELO BEPURE 36 A 39 SOLO NEGR...,25,24,4TO PISO,115,125,12,12.0,...,N22115C-N,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,ZAPATILLA DE DAMA,ZAPATILLAS,HIJA,ZAPATILLA DE DAMA,Dama
2508,2282,LSJ64-JA,ZAPATILLA WEIDE TIPO MONTAÑERA 35 A 38 4 COLORES,25,24,4TO PISO,125,145,36,48.0,...,LSJ64-JA,No identificado,Sin característica identificada,No identificado,Todo el año / Revisar,ZAPATILLA DE DAMA,ZAPATILLAS,HIJA,ZAPATILLA DE DAMA,Dama
2509,2205,EK53151030,ZAPATILLAS DE NIÑO 31 A 36,17,24,CASA CONTENEDOR DICIEMBRE 2025,128,160,24,60.0,...,EK53151030,No identificado,Sin característica identificada,LAS,Todo el año / Revisar,Zapatilla de niño,ZAPATILLAS,HIJA,ZAPATILLA DE NINO,Niño


In [2]:
import pandas as pd

df = pd.read_csv("productos_limpieza_1.csv",encoding = "utf-8-sig")
df.head()

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,...,codigo_limpio,material,caracteristicas,rango_talla,temporada_sugerida,categoria,categoria_principal,tipo_categoria,categoria_limpia,genero_final
0,579,A37012,BOMBER CON CAPUCHA ALGODON S A 2XL,2,1,NaN,188,205,0,NaN,...,A37012,Algodón,Con capucha,S a 2XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
1,613,XL12240M,BRILLO CON CAPUCHA ALGODON M A 3XL,2,1,NaN,218,238,12,NaN,...,XL12240M,Algodón,Con capucha,M a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
2,1484,S-8987-24-2,CASACA VARON CHINO FIBRA CON CAPUCHA ALGODON L...,2,17,.,110,155,36,60.0,...,S-8987-24-2,Algodón,Con capucha,L a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
3,2484,H60222A,CASACA VARON CHINO L A 3XL CON CAPUCHA ALGODON,2,4,.,124,145,36,84.0,...,H60222A,Algodón,Con capucha,L a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
4,1839,9953-24-4,CASACA VARON CHINO XL A 5XL CON CAPUCHA ALGODON,2,17,CASA,125,170,24,120.0,...,9953-24-4,Algodón,Con capucha,XL a 5XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón


In [4]:
df.head()

,id_producto,codigo,producto,id_categoria,id_marca,observacion,Suma de precio_compra,Suma de precio_venta,Suma de stock,Suma de stock_inicial,...,codigo_limpio,material,caracteristicas,rango_talla,temporada_sugerida,categoria,categoria_principal,tipo_categoria,categoria_limpia,genero_final
0,579,A37012,BOMBER CON CAPUCHA ALGODON S A 2XL,2,1,NaN,188,205,0,NaN,...,A37012,Algodón,Con capucha,S a 2XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
1,613,XL12240M,BRILLO CON CAPUCHA ALGODON M A 3XL,2,1,NaN,218,238,12,NaN,...,XL12240M,Algodón,Con capucha,M a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
2,1484,S-8987-24-2,CASACA VARON CHINO FIBRA CON CAPUCHA ALGODON L...,2,17,.,110,155,36,60.0,...,S-8987-24-2,Algodón,Con capucha,L a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
3,2484,H60222A,CASACA VARON CHINO L A 3XL CON CAPUCHA ALGODON,2,4,.,124,145,36,84.0,...,H60222A,Algodón,Con capucha,L a 3XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón
4,1839,9953-24-4,CASACA VARON CHINO XL A 5XL CON CAPUCHA ALGODON,2,17,CASA,125,170,24,120.0,...,9953-24-4,Algodón,Con capucha,XL a 5XL,Todo el año / Revisar,CASACA DE HOMBRE,CASACAS,HIJA,CASACA DE HOMBRE,Varón


In [3]:
import pandas as pd
import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [6]:
patrones_tipo = [
    ("CHALECO", r"\bCHALECO(S)?\b"),
    ("BLAZER", r"\bBLAZER(S)?\b"),
    ("MOCASIN", r"\bMOCASIN(ES)?\b"),
    ("ABRIGO", r"\bABRIGO(S)?\b"),
    ("CASACA", r"\bCASACA(S)?\b"),
    ("LONA", r"\bLONA(S)?\b"),
    ("ZAPATILLA", r"\bZAPATILLA(S)?\b|\bCHIMPUN(ES)?\b"),
]

def detectar_tipo_producto(categoria):
    categoria = normalizar_texto(categoria)

    for tipo, patron in patrones_tipo:
        if re.search(patron, categoria):
            return tipo

    return "NO IDENTIFICADO"

df["tipo_producto"] = df["categoria_limpia"].apply(detectar_tipo_producto)

In [10]:
orden_tallas = {
    "XS": 1,
    "S": 2,
    "M": 3,
    "L": 4,
    "XL": 5,
    "2XL": 6,
    "3XL": 7,
    "4XL": 8,
    "5XL": 9,
    "6XL": 10,
    "7XL": 11,
    "8XL": 12,
    "9XL": 13
}

def extraer_tallas(rango):
    rango = normalizar_texto(rango)

    # Normalizamos algunas formas comunes
    rango = rango.replace("XXL", "2XL")
    rango = rango.replace("XXXL", "3XL")

    tallas = re.findall(r"XS|[2-9]XL|XL|S|M|L", rango)

    return tallas

def clasificar_grupo_talla(row):

    # Solo aplicamos esta regla a casacas
    if normalizar_texto(row["categoria_principal"]) != "CASACAS":
        return "NO APLICA"

    tallas = extraer_tallas(row["rango_talla"])

    if len(tallas) == 0:
        return "NO IDENTIFICADO"

    valores = [orden_tallas[talla] for talla in tallas if talla in orden_tallas]

    if len(valores) == 0:
        return "NO IDENTIFICADO"

    talla_minima = min(valores)

    # Nueva regla:
    # S a 3XL = NORMAL
    # M a 3XL = NORMAL
    # 2XL a 5XL = EXTRA
    # 3XL a 6XL = EXTRA

    if talla_minima >= orden_tallas["2XL"]:
        return "EXTRA"
    else:
        return "NORMAL"

df["grupo_talla_casaca"] = df.apply(clasificar_grupo_talla, axis=1)

In [11]:
df[[
    "codigo",
    "producto_limpio",
    "categoria_limpia",
    "rango_talla",
    "tipo_producto",
    "grupo_talla_casaca"
]].head(30)

,codigo,producto_limpio,categoria_limpia,rango_talla,tipo_producto,grupo_talla_casaca
0,A37012,BOMBER CON CAPUCHA ALGODON S A 2XL,CASACA DE HOMBRE,S a 2XL,CASACA,NORMAL
1,XL12240M,BRILLO CON CAPUCHA ALGODON M A 3XL,CASACA DE HOMBRE,M a 3XL,CASACA,NORMAL
2,S-8987-24-2,CASACA VARON CHINO FIBRA CON CAPUCHA ALGODON L...,CASACA DE HOMBRE,L a 3XL,CASACA,NORMAL
3,H60222A,CASACA VARON CHINO L A 3XL CON CAPUCHA ALGODON,CASACA DE HOMBRE,L a 3XL,CASACA,NORMAL
4,9953-24-4,CASACA VARON CHINO XL A 5XL CON CAPUCHA ALGODON,CASACA DE HOMBRE,XL a 5XL,CASACA,NORMAL
5,PT85062,CASACA VARON YD CON CAPUCHA ALGODON S A 2XL,CASACA DE HOMBRE,S a 2XL,CASACA,NORMAL
6,A11053R,CASACA VARON YD S A 2XL CON CAPUCHA ALGODON,CASACA DE HOMBRE,S a 2XL,CASACA,NORMAL
7,CH-SH7208,CASACAVARON BAIFA CON CAPUCHA ALGODON S A XL,CASACA DE HOMBRE,S a XL,CASACA,NORMAL
8,BD10611,CASACA VARON YD CORTAVIENTO CON CAPUCHA ALGODO...,CASACA DE HOMBRE,S a 3XL,CASACA,NORMAL
9,AE106015Q,CASACA VARON YD CORTAVIENTO GRUESO CON CAPUCHA...,CASACA DE HOMBRE,S a 2XL,CASACA,NORMAL


In [13]:
df["grupo_talla_casaca"].value_counts()

,count
grupo_talla_casaca,
NORMAL,1833
NO APLICA,240
EXTRA,227
NO IDENTIFICADO,209


In [14]:
df["tipo_producto"].value_counts()

,count
tipo_producto,
CASACA,2081
ZAPATILLA,220
CHALECO,123
ABRIGO,49
BLAZER,15
MOCASIN,15
LONA,5
NO IDENTIFICADO,1


In [15]:
df.to_csv("productos_con_tipo_y_talla.csv", index=False, encoding="utf-8-sig")